# Task 2.2: Predicting Rainfall Data

## 2.2.1 Spark Session

In [1]:
import socket
from functools import reduce

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.sql import SparkSession, Window, DataFrame

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F
import seaborn as sns
import pandas as pd

import os
import sys

import time
from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array

## 2.2.2 Reading the Train, Validation and Test Datasets

In [ ]:
train_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/train_parquet/")
val_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/val_parquet/")
test_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/test_parquet/")

## 2.2.3 Training the Model

In [12]:
assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="features",
                            handleInvalid="skip")

gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxDepth=5, maxIter=50, stepSize=0.1, seed=42)

train_v = assembler.transform(train_final)
test_v  = assembler.transform(val_final)

model = gbt.fit(train_v)
preds = model.transform(test_v)

pr  = BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)
roc = BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)

base_rate = preds.select(F.avg("label")).first()[0]
print(f"PR-AUC  {pr:.4f}   (baseline {base_rate:.4f})")
print(f"ROC-AUC {roc:.4f}")

26/10/14 13:46:27 ERROR AsyncEventQueue: Dropping event from queue eventLog. This likely means one of the listeners is too slow and cannot keep up with the rate at which tasks are being started by the scheduler.
26/10/14 13:46:27 WARN AsyncEventQueue: Dropped 1 events from eventLog since the application started.
26/10/14 13:47:27 WARN AsyncEventQueue: Dropped 85107 events from eventLog since Wed Oct 14 13:46:27 UTC 2026.
26/10/14 13:48:27 WARN AsyncEventQueue: Dropped 88289 events from eventLog since Wed Oct 14 13:47:27 UTC 2026.
26/10/14 13:49:27 WARN AsyncEventQueue: Dropped 86128 events from eventLog since Wed Oct 14 13:48:27 UTC 2026.
26/10/14 13:50:27 WARN AsyncEventQueue: Dropped 87939 events from eventLog since Wed Oct 14 13:49:27 UTC 2026.
26/10/14 13:51:27 WARN AsyncEventQueue: Dropped 90869 events from eventLog since Wed Oct 14 13:50:27 UTC 2026.
26/10/14 13:52:27 WARN AsyncEventQueue: Dropped 92180 events from eventLog since Wed Oct 14 13:51:27 UTC 2026.
26/10/14 13:53:27 WA

KeyboardInterrupt: 

26/10/14 14:22:39 WARN AsyncEventQueue: Dropped 76306 events from eventLog since Wed Oct 14 14:21:39 UTC 2026.
26/10/14 14:23:39 WARN AsyncEventQueue: Dropped 74645 events from eventLog since Wed Oct 14 14:22:39 UTC 2026.
26/10/14 14:24:39 WARN AsyncEventQueue: Dropped 75262 events from eventLog since Wed Oct 14 14:23:39 UTC 2026.
26/10/14 14:25:39 WARN AsyncEventQueue: Dropped 74677 events from eventLog since Wed Oct 14 14:24:39 UTC 2026.
26/10/14 14:26:39 WARN AsyncEventQueue: Dropped 72144 events from eventLog since Wed Oct 14 14:25:39 UTC 2026.
26/10/14 14:27:39 WARN AsyncEventQueue: Dropped 78436 events from eventLog since Wed Oct 14 14:26:39 UTC 2026.
26/10/14 14:28:39 WARN AsyncEventQueue: Dropped 71560 events from eventLog since Wed Oct 14 14:27:39 UTC 2026.


In [ ]:
# feature importances
for name, imp in sorted(zip(feature_cols, model.featureImportances.toArray()),
                        key=lambda x: -x[1]):
    print(f"{imp:.4f}  {name}")

# persistence baseline 
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))
naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

0.4837  rain_cumsum_30m
0.3491  reading_value
0.0506  hour
0.0499  rain_cumsum_1h
0.0244  month
0.0229  lag_5m
0.0130  lag_30m
0.0062  lag_1h
0.0001  location_longitude
0.0000  location_latitude


Row(avg(CAST((naive = label) AS INT))=0.9670901466257842)


In [18]:
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))

naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

+-----+-----+-------+
|naive|label|  count|
+-----+-----+-------+
|    0|    0|6006697|
|    0|    1|  93060|
|    1|    0| 116011|
|    1|    1| 137071|
+-----+-----+-------+



In [19]:
preds.groupBy("prediction", "label").count().orderBy("prediction", "label").show()

+----------+-----+-------+
|prediction|label|  count|
+----------+-----+-------+
|       0.0|    0|6097169|
|       0.0|    1| 121179|
|       1.0|    0|  25539|
|       1.0|    1| 108952|
+----------+-----+-------+



In [21]:
base_rate = train.select(F.avg("label")).first()[0]

shuffled = train.withColumn("label",
    F.col("label") * 0 + (F.rand(seed=7) < base_rate).cast("int"))

m2 = gbt.fit(assembler.transform(shuffled))
p2 = m2.transform(assembler.transform(test))
print(BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.1974824343394064


In [22]:
print(BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.5637655553184941


26/10/11 16:21:28 ERROR TransportResponseHandler: Still have 1 requests outstanding when connection from /10.148.0.33:45252 is closed
26/10/11 16:21:28 ERROR TransportClient: Failed to send RPC RPC 8693474043611835936 to /10.148.0.33:45252: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException: null
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source) ~[netty-transport-4.1.100.Final.jar:4.1.100.Final]
26/10/11 16:21:28 WARN BlockManagerMasterEndpoint: Error trying to remove RDD 2211 from block manager BlockManagerId(31, rainfall-dev-w-0.asia-southeast1-b.c.dsa5208-rainfall-sixing.internal, 40695, None)
java.io.IOException: Connection from /10.148.0.33:45252 closed
	at org.apache.spark.network.client.TransportResponseHandler.channelInactive(TransportResponseHandler.java:147) ~[spark-network-common_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.server.TransportChannelHandler.channelInactive(Trans